# Three-step GC comparison on a circle

This study compares the same circular boundary integrated up to $4\pi$ using three BM4 step sizes: $\pi/40$, $\pi/80$, and $\pi/160$. The circle contains 16 points, twice as many as the previous diagnostic contour.

The single animation assigns one color to each step size and synchronizes four views:

- all three contours over the effective potential;
- the relative area error $\varepsilon_A$;
- the relative symplectic defect $\|DG^T\Omega DG-\Omega\|_F/\|\Omega\|_F$;
- the relative copy separation $\|z_1-z_2\|_2/\|(z_1+z_2)/2\|_2$.

States are saved every $\pi/8$. This interval contains exactly 5, 10, and 20 steps, respectively, so saving does not alter the step sizes being compared.

In [1]:
from pathlib import Path
import sys

import matplotlib as mpl
from matplotlib import animation as mpl_animation
import numpy as np
from IPython.display import HTML, display


def find_project_root(start):
    """Find the repository from either Jupyter's root or this folder."""
    location = Path(start).resolve()
    for candidate in (location, *location.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError('Could not find the project root.')


def display_animation(animation):
    """Display a compact HTML5 animation, with a JavaScript fallback."""
    mpl.rcParams['animation.embed_limit'] = 100.0
    if mpl_animation.writers.is_available('ffmpeg'):
        display(HTML(animation.to_html5_video()))
    else:
        display(HTML(animation.to_jshtml(default_mode='once')))


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from classes import Area, Potential, SystemGC
from research.projection import ProjectedSymplecticityAreaObserver

NOTEBOOK_PATH = (
    PROJECT_ROOT
    / 'notebooks'
    / 'developements'
    / 'study_gc_area_and_projected_symplecticity.ipynb'
)

In [2]:
# False preserves all three scales but reduces the time span and point count for validation.
full_study = True
step_specs = (
    (r'$\Delta t=\pi/40$', np.pi / 40),
    (r'$\Delta t=\pi/80$', np.pi / 80),
    (r'$\Delta t=\pi/160$', np.pi / 160),
)
save_interval = np.pi / 8
comparison_t_span = (
    (0.0, 4 * np.pi) if full_study else (0.0, np.pi / 4)
)
n_output_samples = int(round(
    (comparison_t_span[1] - comparison_t_span[0]) / save_interval
)) + 1
circle_points = 16 if full_study else 8
coupling_frequency = 0.0
rho = 0.3

potential = Potential.random(
    A=0.7,
    M=25,
    nx=64,
    ny=64,
    seed=27,
    interpolation_order=5,
)
center = (
    potential.grid.xmin + potential.grid.period / 2,
    potential.grid.ymin + potential.grid.period / 2,
)
circle = Area.circle(
    center=center,
    radius=0.5,
    points=circle_points,
    rho=rho,
)
system = SystemGC(
    potential,
    circle,
    coupling_frequency=coupling_frequency,
)
print(
    f'Circle: {circle_points} points; t={comparison_t_span}; '
    f'{n_output_samples} saved states'
)

Circle: 16 points; t=(0.0, 12.566370614359172); 33 saved states


## Integrations and observations

Each observer saves one sample per $\pi/8$ interval. Blocks are written under `outputs/developements/study_gc_area_and_projected_symplecticity/<date>/`, with a distinct name for each step size.

In [3]:
comparison_solutions = {}
comparison_records = {}
comparison_output_directories = {}

for label, step in step_specs:
    # These exact integer ratios keep all diagnostics on the save grid.
    diagnostic_stride = int(round(save_interval / step))
    step_tag = f'{step:.8f}'.replace('.', 'p')
    with ProjectedSymplecticityAreaObserver(
        notebook_path=NOTEBOOK_PATH,
        area=circle,
        period=potential.grid.period,
        project_root=PROJECT_ROOT,
        block_name=f'circle_comparison_step_{step_tag}',
        record_every=diagnostic_stride,
        chunk_size=16,
        verbose=False,
        metadata={
            'geometry': 'circle',
            'circle_points': circle_points,
            'integration_step': step,
            'coupling_frequency': coupling_frequency,
            'rho': rho,
            'potential_seed': 27,
        },
    ) as observer:
        solution = system.simulate(
            step=step,
            t_span=comparison_t_span,
            n_output_samples=n_output_samples,
            check_energy=False,
            progress=full_study,
            stage_observer=observer,
        )

    comparison_solutions[label] = solution
    comparison_records[label] = observer.records
    comparison_output_directories[label] = observer.output_directory
    print(
        f'{label}: {solution.n_steps} BM4 steps, '
        f'{len(observer.records)} observations -> {observer.output_directory}'
    )

SystemGC [==============================] 100.0% (174/174, t=12.5664)


$\Delta t=\pi/40$: 174 BM4 steps, 36 observations -> /home/juan/Proyectos/GC2D_intranet/outputs/developements/study_gc_area_and_projected_symplecticity/2026-07-21


SystemGC [==============================] 100.0% (334/334, t=12.5664)


$\Delta t=\pi/80$: 334 BM4 steps, 35 observations -> /home/juan/Proyectos/GC2D_intranet/outputs/developements/study_gc_area_and_projected_symplecticity/2026-07-21


SystemGC [==============================] 100.0% (654/654, t=12.5664)


$\Delta t=\pi/160$: 654 BM4 steps, 34 observations -> /home/juan/Proyectos/GC2D_intranet/outputs/developements/study_gc_area_and_projected_symplecticity/2026-07-21


In [4]:
header = (
    'step',
    'max |area error|',
    'max symplectic defect',
    'max relative separation',
)
print(f'{header[0]:>22} {header[1]:>20} {header[2]:>26} {header[3]:>26}')
for label, _step in step_specs:
    records = comparison_records[label]
    print(
        f'{label:>22} '
        f'{max(abs(record.relative_area_error) for record in records):20.8e} '
        f'{max(record.relative_defect for record in records):26.8e} '
        f'{max(record.relative_copy_separation for record in records):26.8e}'
    )

                  step     max |area error|      max symplectic defect    max relative separation
     $\Delta t=\pi/40$       1.72107685e-02             4.03881412e-09             1.13710857e-08
     $\Delta t=\pi/80$       1.74432630e-02             4.34023314e-09             5.07920429e-10
    $\Delta t=\pi/160$       1.74474736e-02             3.98679682e-09             2.99274916e-11


## Comparative animation

The same color identifies a step size in all four panels. The curve markers indicate the most recent data point available at the frame time.

In [5]:
diagnostic_times = {
    label: np.asarray([record.time for record in records])
    for label, records in comparison_records.items()
}
relative_symplecticity_errors = {
    label: np.asarray([record.relative_defect for record in records])
    for label, records in comparison_records.items()
}
relative_copy_separations = {
    label: np.asarray([record.relative_copy_separation for record in records])
    for label, records in comparison_records.items()
}

comparison_animation = system.animate_area_comparison(
    comparison_solutions,
    diagnostic_times=diagnostic_times,
    relative_symplecticity_errors=relative_symplecticity_errors,
    relative_copy_separations=relative_copy_separations,
    frames=None,
    interval=120,
)
display_animation(comparison_animation)

## Interpretation

The comparison distinguishes step-size convergence from geometric resolution: all three calculations use exactly the same 16-vertex circle, potential, initial condition, and observation times. Therefore, differences between the curves arise from the BM4 step size, while the common polygonal error comes from representing the circle with a finite number of points.